In [2]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.9 MB/s eta 0:00:00


In [3]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stopwords_list = set(stopword_factory.get_stop_words())

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Library berhasil dimuat.")

Library berhasil dimuat.


In [4]:
documents = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama.",
    "Jaringan komputer menghubungkan banyak perangkat agar dapat bertukar data secara cepat.",
    "Kecerdasan buatan memungkinkan komputer belajar dari data tanpa diprogram secara eksplisit.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen relevan dari data komputer yang besar.",
]

doc_labels = [f"D{i+1}" for i in range(len(documents))]

for label, doc in zip(doc_labels, documents):
    print(f"{label}: {doc}")

D1: Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama.
D2: Jaringan komputer menghubungkan banyak perangkat agar dapat bertukar data secara cepat.
D3: Kecerdasan buatan memungkinkan komputer belajar dari data tanpa diprogram secara eksplisit.
D4: Sistem temu kembali informasi membantu pengguna menemukan dokumen relevan dari data komputer yang besar.


In [5]:
def simple_preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_list]
    return tokens

tokenized_docs = [simple_preprocess(doc) for doc in documents]
for label, tokens in zip(doc_labels, tokenized_docs):
    print(f"{label}: {tokens}")

D1: ['sistem', 'komputer', 'terdiri', 'perangkat', 'keras', 'perangkat', 'lunak', 'bekerja', 'sama']
D2: ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'bertukar', 'data', 'cepat']
D3: ['kecerdasan', 'buatan', 'memungkinkan', 'komputer', 'belajar', 'data', 'diprogram', 'eksplisit']
D4: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan', 'data', 'komputer', 'besar']


In [6]:
# Kumpulkan seluruh term unik (vocabulary)
vocab = sorted(set(term for tokens in tokenized_docs for term in tokens))

bow_data = {
    label: {term: tokens.count(term) for term in vocab}
    for label, tokens in zip(doc_labels, tokenized_docs)
}

bow_df = pd.DataFrame(bow_data).T  # baris = dokumen, kolom = term
bow_df = bow_df[vocab]
bow_df

,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,2,0,1,1,0,1
D2,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0
D3,0,0,1,0,0,1,0,1,1,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
D4,0,0,0,0,1,0,0,1,0,1,0,1,0,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0


In [7]:
tf_manual = bow_df.div(bow_df.sum(axis=1), axis=0)
tf_manual

,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0.000,0.111111,0.000,0.000,0.000000,0.000,0.000,0.000000,0.000,0.000000,0.000,0.000000,0.000,0.000,0.111111,0.111111,0.111111,0.000000,0.000,0.000000,0.000,0.000000,0.222222,0.000000,0.111111,0.111111,0.000000,0.111111
D2,0.125,0.000000,0.000,0.125,0.000000,0.000,0.125,0.125000,0.000,0.000000,0.000,0.000000,0.125,0.000,0.000000,0.125000,0.000000,0.000000,0.000,0.000000,0.125,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000
D3,0.000,0.000000,0.125,0.000,0.000000,0.125,0.000,0.125000,0.125,0.000000,0.125,0.000000,0.000,0.125,0.000000,0.125000,0.000000,0.000000,0.125,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D4,0.000,0.000000,0.000,0.000,0.090909,0.000,0.000,0.090909,0.000,0.090909,0.000,0.090909,0.000,0.000,0.000000,0.090909,0.000000,0.090909,0.000,0.090909,0.000,0.090909,0.000000,0.090909,0.000000,0.090909,0.090909,0.000000


In [8]:
N = len(documents)

df_series = (bow_df > 0).sum(axis=0)  # jumlah dokumen yang memuat term
idf_series = np.log((1 + N) / (1 + df_series)) + 1

idf_df = pd.DataFrame({"DF": df_series, "IDF": idf_series})
idf_df

,DF,IDF
banyak,1,1.916291
bekerja,1,1.916291
belajar,1,1.916291
bertukar,1,1.916291
besar,1,1.916291
buatan,1,1.916291
cepat,1,1.916291
data,3,1.223144
diprogram,1,1.916291
dokumen,1,1.916291


In [9]:
tfidf_manual_raw = tf_manual.mul(idf_series, axis=1)

# Normalisasi L2 per dokumen (baris)
l2_norm = np.sqrt((tfidf_manual_raw ** 2).sum(axis=1))
tfidf_manual = tfidf_manual_raw.div(l2_norm, axis=0)

tfidf_manual.round(4)

,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0.0000,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3454,0.1803,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.5447,0.0000,0.3454,0.2723,0.0000,0.3454
D2,0.3984,0.0000,0.0000,0.3984,0.0000,0.0000,0.3984,0.2543,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.0000,0.2079,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.3141,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2470,0.3869,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2019,0.0000,0.0000,0.3869,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.3279,0.0000,0.0000,0.2093,0.0000,0.3279,0.0000,0.3279,0.0000,0.0000,0.0000,0.1711,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.2585,0.3279,0.0000


In [10]:
# Gunakan dokumen yang sudah di-preprocess (join kembali jadi string)
preprocessed_texts = [" ".join(tokens) for tokens in tokenized_docs]

vectorizer = TfidfVectorizer()
tfidf_sklearn_matrix = vectorizer.fit_transform(preprocessed_texts)

tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn_matrix.toarray(),
    index=doc_labels,
    columns=vectorizer.get_feature_names_out(),
)
tfidf_sklearn.round(4)

,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0.0000,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3454,0.1803,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.5447,0.0000,0.3454,0.2723,0.0000,0.3454
D2,0.3984,0.0000,0.0000,0.3984,0.0000,0.0000,0.3984,0.2543,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.0000,0.2079,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.3141,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2470,0.3869,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2019,0.0000,0.0000,0.3869,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.3279,0.0000,0.0000,0.2093,0.0000,0.3279,0.0000,0.3279,0.0000,0.0000,0.0000,0.1711,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.2585,0.3279,0.0000


In [11]:
# Samakan urutan kolom untuk perbandingan
common_cols = sorted(set(tfidf_manual.columns) & set(tfidf_sklearn.columns))

comparison = tfidf_manual[common_cols].round(4) - tfidf_sklearn[common_cols].round(4)
print("Selisih maksimum antara TF-IDF manual dan scikit-learn:", comparison.abs().values.max())

print("\nTF-IDF Manual:")
display(tfidf_manual[common_cols].round(4))

print("\nTF-IDF scikit-learn:")
display(tfidf_sklearn[common_cols].round(4))

Selisih maksimum antara TF-IDF manual dan scikit-learn: 0.0

TF-IDF Manual:


,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0.0000,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3454,0.1803,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.5447,0.0000,0.3454,0.2723,0.0000,0.3454
D2,0.3984,0.0000,0.0000,0.3984,0.0000,0.0000,0.3984,0.2543,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.0000,0.2079,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.3141,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2470,0.3869,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2019,0.0000,0.0000,0.3869,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.3279,0.0000,0.0000,0.2093,0.0000,0.3279,0.0000,0.3279,0.0000,0.0000,0.0000,0.1711,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.2585,0.3279,0.0000



TF-IDF scikit-learn:


,banyak,bekerja,belajar,bertukar,besar,buatan,cepat,data,diprogram,dokumen,eksplisit,informasi,jaringan,kecerdasan,keras,komputer,lunak,membantu,memungkinkan,menemukan,menghubungkan,pengguna,perangkat,relevan,sama,sistem,temu,terdiri
D1,0.0000,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3454,0.1803,0.3454,0.0000,0.0000,0.0000,0.0000,0.0000,0.5447,0.0000,0.3454,0.2723,0.0000,0.3454
D2,0.3984,0.0000,0.0000,0.3984,0.0000,0.0000,0.3984,0.2543,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.0000,0.2079,0.0000,0.0000,0.0000,0.0000,0.3984,0.0000,0.3141,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2470,0.3869,0.0000,0.3869,0.0000,0.0000,0.3869,0.0000,0.2019,0.0000,0.0000,0.3869,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.3279,0.0000,0.0000,0.2093,0.0000,0.3279,0.0000,0.3279,0.0000,0.0000,0.0000,0.1711,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.3279,0.0000,0.2585,0.3279,0.0000


In [12]:
top_terms = tfidf_sklearn.idxmax(axis=1)
top_scores = tfidf_sklearn.max(axis=1)

top_df = pd.DataFrame({
    "Dokumen": doc_labels,
    "Term Tertinggi": top_terms.values,
    "Bobot TF-IDF": top_scores.round(4).values,
})
top_df

,Dokumen,Term Tertinggi,Bobot TF-IDF
0,D1,perangkat,0.5447
1,D2,banyak,0.3984
2,D3,belajar,0.3869
3,D4,besar,0.3279


Analisis singkat :
Term dengan bobot TF-IDF tertinggi pada masing-masing dokumen umumnya adalah term yang jarang muncul di dokumen lain (DF rendah, sehingga IDF tinggi) namun cukup sering muncul pada dokumen tersebut (TF relatif tinggi) — misalnya term-term spesifik seperti *"jaring"* pada dokumen tentang jaringan komputer atau *"cerdas"/"buat"* pada dokumen tentang kecerdasan buatan. Term seperti *"komputer"* yang muncul di hampir semua dokumen justru memperoleh bobot IDF (dan karenanya TF-IDF) yang lebih rendah, karena kurang membedakan satu dokumen dari dokumen lainnya. Hal ini sesuai dengan prinsip dasar TF-IDF, yaitu memberi bobot tinggi pada term yang sering muncul di suatu dokumen tetapi jarang muncul di seluruh koleksi, sehingga term tersebut lebih representatif dan diskriminatif untuk membedakan topik/isi dokumen pada sistem temu kembali informasi.